# 02 · Alertas de inventario: un mes, todos los meses y validación

Reto 1 — Monitoreo de saldo de inventarios, FEMSA Salud Chile (FCV).

Este notebook usa el paquete `reto_ma2004b` (carpeta `src/reto_ma2004b/`), que reúne en funciones todo lo que antes vivía en celdas:

| Módulo | Qué hace |
|---|---|
| `carga` | Lee y limpia Kardex, balanza, recetario (TD DETA) y tipo de cambio |
| `reglas` | Alertas por partida: réplica de las fórmulas de Excel, semáforo en días y riesgo P0/P1 |
| `indicadores` | Conciliación, variación mensual, KPIs, tabla de alertas y transiciones |
| `validacion` | Compara contra los valores que calculó Excel |
| `pipeline` | Script ejecutable que procesa uno o varios meses y deja las tablas de Power BI |

Contenido:
1. Un mes paso a paso (enero).
2. Todos los meses con el pipeline y validación contra Excel.
3. Conciliación contra el dashboard de la propuesta.
4. Impacto de la corrección de fechas que hizo FEMSA.
5. Semáforos, riesgo por política de canje y alertas.
6. Transiciones entre rangos de antigüedad (base del modelo de Markov).

Requisito: Excel desencriptados en `data/raw/` (`dvc pull -r storage` y `./scripts/dvc_decrypt.sh`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import pandas as pd

from reto_ma2004b import carga, indicadores, reglas, validacion
from reto_ma2004b.pipeline import ejecutar, raiz_del_repo

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 40)

ROOT = raiz_del_repo()
RAW = ROOT / "data" / "raw"
SALIDA = ROOT / "data" / "processed" / "powerbi"
FIGURAS = ROOT / "reports" / "figures"
FIGURAS.mkdir(parents=True, exist_ok=True)

archivos = carga.ubicar_archivos([RAW])
list(archivos["meses"]), archivos["tipo_cambio"].name

## 1. Un mes paso a paso: enero

Primero el proceso con funciones sueltas, para ver qué hace cada paso. El pipeline de la sección 2 encadena exactamente estas mismas funciones.

In [ ]:
rutas = archivos["meses"]["202601"]
kardex = carga.leer_kardex(rutas["kardex"])            # limpieza: solo columnas de entrada, sin subtotales
detalle = reglas.construir_columnas(kardex)            # todas las alertas por partida
saldo = carga.saldo_cuenta(carga.leer_balanza(rutas["balanza"]))
recetarios = carga.leer_recetarios([d["kardex"] for d in archivos["meses"].values()] + archivos["otros"])
tc = carga.leer_tipo_cambio(archivos["tipo_cambio"])

print(f"{len(detalle):,} partidas, {detalle.shape[1]} columnas")
detalle[["CODIGO", "STOCK", "COSTO", "VALORIZACION", "FULCOM", "MESES_ANTIG", "SEMAFORO_MESES",
         "DIAS_ANTIG", "SEMAFORO_DIAS", "POLITICA_CANJE", "RIESGO_POLITICA", "ESTADO_GENERAL"]].head()

In [ ]:
resumen_enero = indicadores.resumen_mes(detalle, saldo, recetarios.get("202601"), tc.get("202601"))
pd.Series(resumen_enero)[["SALDO_GL", "SALDO_GL_MXN", "TOTAL_KARDEX", "RECETARIO", "DIF_KARDEX_GL", "DIF_NO_EXPLICADA",
                          "ESTADO_CONCILIACION", "SEMAFORO_CONCILIACION", "SKUS_UNICOS",
                          "PARTIDAS_MAS_1_ANIO", "PARTIDAS_ROJO_MESES", "PARTIDAS_ROJO_DIAS", "PARTIDAS_RIESGO_ALTO"]]

La conciliación se lee en tres pasos: el Kardex queda 508.6 millones por debajo del GL (`DIF_KARDEX_GL`), el recetario explica esa diferencia y lo que queda sin explicar son 43 centavos (`DIF_NO_EXPLICADA`), por eso el semáforo de conciliación sale en verde. FEMSA confirmó que el recetario no tiene integración: es un valor capturado en la tabla TD DETA.

## 2. Todos los meses con el pipeline

`ejecutar` es la misma función que corre el script (`uv run reto-ma2004b`). Recibe carpetas o archivos, junta todos los meses, construye las alertas y escribe las tablas de Power BI en `data/processed/powerbi/`. Con `validar=True` compara cada mes contra los valores que guardó Excel.

In [ ]:
tablas = ejecutar([RAW], SALIDA, validar=True)

In [ ]:
v = tablas["validacion"]
v.pivot_table(index="COLUMNA", columns="MES", values="PCT_COINCIDENCIA").style.format("{:.4%}")

In [ ]:
v.loc[v["NA_EN_EXCEL"] > 0, ["MES", "COLUMNA", "PARTIDAS", "COINCIDEN", "DIFIEREN", "NA_EN_EXCEL"]]

En los siete meses todas las columnas coinciden con Excel en el 100% de las partidas, salvo las que Excel deja en `#N/A` (dos partidas en abril, junio y julio, sin unidad de conversión). Ahí la diferencia es intencional: Excel propaga el error y la partida queda sin estado; el código la marca como `INVÁLIDA` y la manda a revisión, como recomienda la hoja de criterios de la propuesta ("crear estado ERROR DE DATOS y mantenerlas visibles"). No hay ninguna otra diferencia.

## 3. Conciliación contra el dashboard de la propuesta

El dashboard de julio de la propuesta (`Dashboard Inventario Julio`, fila 5) usa el mismo saldo GL, el mismo total del Kardex y el mismo recetario. Esos valores no dependen de las fechas, así que deben coincidir también con los datos corregidos.

In [ ]:
propuesta = next(p for p in archivos["otros"] if "propuesta" in p.name.lower())
ws = openpyxl.load_workbook(propuesta, read_only=True, data_only=True)["Dashboard Inventario Julio"]
celda = {(i, j): val for i, fila in enumerate(ws.iter_rows(max_row=6, values_only=True), start=1)
         for j, val in enumerate(fila, start=1)}
julio = tablas["resumen_mensual"].set_index("MES_ORDEN").loc[202607]

pd.DataFrame({
    "excel": [celda[5, 1], celda[5, 4], celda[5, 7], celda[5, 10], celda[5, 12]],
    "python": [julio["SALDO_GL"], julio["INVENTARIO_CONCILIABLE"], julio["DIF_NO_EXPLICADA"],
               julio["ESTADO_CONCILIACION"], julio["SKUS_UNICOS"]],
}, index=["Saldo GL (A5)", "Kardex + recetario (D5)", "Diferencia (G5)", "Estado (J5)", "SKU únicos (L5)"])

## 4. Impacto de la corrección de fechas

La propuesta se construyó con la primera versión de los Kardex, donde las FULCOM estaban recorridas 113 días hacia adelante (por eso marcaba 10,624 "fechas FULCOM futuras" como crítico). Con los datos corregidos el inventario se ve más envejecido: la conciliación no cambia, pero el semáforo sí.

In [ ]:
ws_crit = openpyxl.load_workbook(propuesta, read_only=True, data_only=True)["criterio de alertas "]
crit = {i: [c for c in fila if c is not None] for i, fila in enumerate(ws_crit.iter_rows(max_row=30, values_only=True), start=1)}
comparativo = pd.DataFrame({
    "propuesta (fechas recorridas)": [crit[23][2], crit[24][2], crit[25][2], crit[14][2]],
    "datos corregidos": [julio["PARTIDAS_VERDE_MESES"], julio["PARTIDAS_AMBAR_MESES"], julio["PARTIDAS_ROJO_MESES"],
                         julio["PARTIDAS_FULCOM_FUTURA"]],
}, index=["Verde (≤ 6 meses)", "Ámbar (7–12 meses)", "Rojo (> 12 meses)", "FULCOM posterior al cierre"])
comparativo["cambio"] = comparativo["datos corregidos"] - comparativo["propuesta (fechas recorridas)"]
comparativo

En julio las partidas en rojo pasan de 1,906 a 2,589 (+36%) y las fechas posteriores al cierre desaparecen. Es el efecto que se anticipó en el primer avance al restar los 113 días.

## 5. Semáforos, riesgo por política de canje y alertas

In [ ]:
r = tablas["resumen_mensual"]
r.set_index("MES_ETIQUETA")[[
    "PARTIDAS_VERDE_MESES", "PARTIDAS_AMBAR_MESES", "PARTIDAS_ROJO_MESES",
    "PARTIDAS_VERDE_DIAS", "PARTIDAS_AMBAR_DIAS", "PARTIDAS_ROJO_DIAS",
    "PARTIDAS_RIESGO_ALTO", "VALOR_RIESGO_ALTO", "PARTIDAS_MAS_1_ANIO", "VARIACION_PCT", "ALERTA_VARIACION",
]]

- Con la política en días (hoja de criterios) el rojo es mayor que con la de meses, porque corta en 180 días en lugar de 12 meses. En julio: 3,731 contra 2,589 partidas.
- `RIESGO_ALTO` son las partidas en rojo sin canje (P0), las que la propuesta señala como prioridad: entre 2,246 y 2,653 por mes.
- La variación mensual solo se activa en mayo (−7.35%).

In [ ]:
a = tablas["alertas"]
a.pivot_table(index=["TIPO_ALERTA", "NIVEL"], columns="MES", values="VALOR_CLP", aggfunc="size", fill_value=0)

### Un error de captura en junio

La fecha de última compra de un SKU solo puede avanzar o quedarse igual de un mes a otro. El pipeline marca `FULCOM_RETROCEDE` cuando retrocede, y en junio aparece un grupo de SKU cuya FULCOM pasó del 01/05/2026 (mayo) al 01/05/2025: exactamente un año antes.

In [ ]:
p = tablas["partidas"]
ret = p[p["FULCOM_RETROCEDE"] == 1]
print(f"{len(ret):,} partidas, {ret['VALORIZACION'].sum():,.0f} CLP, todas en el mes:", ret["MES"].dt.strftime("%Y-%m").unique())
ejemplo = ret["CODIGO"].iloc[0]
p.loc[p["CODIGO"] == ejemplo, ["MES", "CODIGO", "FULCOM", "DIAS_ANTIG", "SEMAFORO_DIAS", "RIESGO_POLITICA"]]

In [ ]:
r.set_index("MES_ETIQUETA")[["PARTIDAS_MAS_1_ANIO", "VALOR_MAS_1_ANIO", "PARTIDAS_FULCOM_RETROCEDE",
                              "PARTIDAS_FULCOM_SOSPECHOSA", "VALOR_FULCOM_SOSPECHOSA"]]

Son 591 partidas por cerca de 1,000 millones CLP que en junio pasan de recién compradas a más de un año. Explican el salto del valor con más de un año (de 302 millones en mayo a 1,291 millones en junio). En julio, 308 de ellas siguen con la fecha de 2025 (~482 millones CLP; `FULCOM_SOSPECHOSA`), mientras que las demás se volvieron a comprar. Todo apunta a un error de año al capturar o al anonimizar; el tablero lo muestra como alerta de calidad de datos y conviene confirmarlo con FEMSA.

## 6. Transiciones entre rangos de antigüedad

Para cada SKU se registra su rango en un mes y en el siguiente ("Sale" si deja de estar en el Kardex). Normalizando por fila se obtiene una primera matriz de transición mensual, que es la base del modelo de Markov de la siguiente etapa. El 1% que salta de "0-30 días" a "más de 365 días" corresponde al error de junio descrito arriba; antes de estimar el modelo habrá que decidir cómo tratarlo.

In [ ]:
t = tablas["transiciones"]
orden = [e for e in tablas["dim_rango"].sort_values("RANGO_ORDEN")["RANGO_ANTIG"] if e != "Sin fecha"]
matriz = pd.crosstab(t["RANGO_ANTIG"], t["RANGO_SIGUIENTE"], normalize="index").reindex(index=orden, columns=orden + ["Sale"]).fillna(0)
(matriz * 100).style.format("{:.1f}%").background_gradient(cmap="Blues", axis=None)

### Figuras

In [ ]:
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
AZUL, CRITICO = "#2a78d6", "#d03b3b"
x = np.arange(len(r))
fig, ax = plt.subplots(figsize=(6.2, 2.8))
saldo = r["SALDO_GL"] / 1e9
alerta = (r["ALERTA_VARIACION"] == "ALERTA").to_numpy()
ax.plot(x, saldo, color=AZUL, linewidth=2)
ax.scatter(x[~alerta], saldo[~alerta], s=36, color=AZUL, edgecolor="white", zorder=3)
ax.scatter(x[alerta], saldo[alerta], s=48, color=CRITICO, edgecolor="white", zorder=3, label="Variación ≥ 5% (alerta)")
for i, (val, var) in enumerate(zip(saldo, r["VARIACION_PCT"])):
    if pd.notna(var):
        ax.text(i, val + (0.9 if var >= 0 else -0.9), f"{var:+.1%}", ha="center", va="bottom" if var >= 0 else "top", fontsize=8)
ax.set_xticks(x, r["MES_ETIQUETA"].str[:3])
ax.set_ylim(110, 132)
ax.set_ylabel("Saldo GL, miles de millones CLP")
ax.yaxis.grid(True, color="#e4e3dd", linewidth=0.6)
ax.legend(frameon=False, loc="lower left")
fig.tight_layout()
fig.savefig(FIGURAS / "saldo_gl_mensual.pdf", metadata={"CreationDate": None})
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 2.8))
ax.plot(x, r["PARTIDAS_ROJO_DIAS"], color=CRITICO, linewidth=2, marker="o", label="Rojo, política en días (> 180 días)")
ax.plot(x, r["PARTIDAS_ROJO_MESES"], color="#8a5a00", linewidth=2, marker="s", label="Rojo, fórmulas (> 12 meses)")
ax.plot(x, r["PARTIDAS_RIESGO_ALTO"], color="#5f5e5a", linewidth=2, linestyle="--", marker="^", label="Riesgo alto (rojo P0)")
ax.set_xticks(x, r["MES_ETIQUETA"].str[:3])
ax.set_ylim(0, 4500)
ax.set_ylabel("Partidas")
ax.yaxis.grid(True, color="#e4e3dd", linewidth=0.6)
ax.legend(frameon=False, loc="lower left", fontsize=8)
fig.tight_layout()
fig.savefig(FIGURAS / "partidas_rojo_mensual.pdf", metadata={"CreationDate": None})
plt.show()

## Conclusión

- El código modular reproduce las fórmulas de Excel en las 107,302 partidas de enero a julio; las únicas diferencias son las 12 celdas que Excel deja en `#N/A`, que el código mantiene visibles a propósito.
- La conciliación cuadra los siete meses con diferencias de centavos una vez sumado el recetario.
- Con los datos corregidos, el rojo de julio sube de 1,906 a 2,589 partidas (fórmulas) y a 3,731 con la política en días; el riesgo alto (rojo P0) es de 2,445 partidas por 728 millones CLP.
- En junio, 591 partidas (~1,000 millones CLP) tienen la fecha de compra un año antes que en mayo, y 308 conservan esa fecha en julio; se reporta como posible error de captura.
- Las tablas de `data/processed/powerbi/` alimentan el dashboard, y `transiciones.csv` deja lista la entrada del modelo de Markov.